# AI-assisted workflow

This notebook repeats the workflow of this week’s lesson, but this time an
AI assistant helps us write the code. The division of labour is simple:

- **You decide the GIS part**: which data, which coordinate reference system,
  which spatial predicate, which classes. You are the GIS expert, and these
  decisions go *into* your prompt.
- **The AI assistant helps with the Python part**: boilerplate, syntax you
  don’t remember, restructuring code, explaining error messages.

The responses shown below are *examples*. If you run the same prompts yourself,
you will get different code, and that is fine. What matters is that you can
read the code, run it, and check the result.

This week, we use an AI assistant to help with a spatial join: how many of the
addresses we geocoded in the lesson are located in each city district of
Helsinki?

> **Habit of the week: share the structure, not the data**
>
> Whatever you paste into an online AI assistant leaves your computer. Our
> sample addresses are public, but in real projects addresses often belong to
> people (customers, patients, survey respondents) and are *personal data*.
> The assistant usually doesn’t need your data to write code: the column
> names, data types, CRS and one or two **made-up** example rows are enough.

## Your decisions

- Points: the addresses we geocoded in the lesson ([Geocoding in geopandas](geocoding-in-geopandas)), saved in `data/addresses.gpkg`.
- Polygons: Helsinki city districts (2021).
- Spatial predicate: an address counts for a district if it is **within** the district polygon.
- Districts without any address should stay in the result (with a count of 0).

In [ ]:
import pathlib
NOTEBOOK_PATH = pathlib.Path().resolve()
DATA_DIRECTORY = NOTEBOOK_PATH / "data"

## Step 1: describe your data, without sharing it

In [ ]:
import geopandas

addresses = geopandas.read_file(DATA_DIRECTORY / "addresses.gpkg", layer="addresses")
city_districts = geopandas.read_file(
    DATA_DIRECTORY / "helsinki_city_districts" / "helsinki_city_districts_2021.gpkg"
)

print(addresses.dtypes)
print(f"CRS: {addresses.crs}")

print(city_districts.dtypes)
print(f"CRS: {city_districts.crs}")

This is all the assistant needs to know about our data: column names, data
types and the CRS. If an example helps, invent one (such as `"Esimerkkikatu 1,
00100 Helsinki"`) instead of copying real rows.

## Step 2: count addresses per district

**Prompt:**

```text
I use geopandas 1.x. I have two GeoDataFrames:
- `addresses`: points, columns "address", "geometry", EPSG:4326
- `city_districts`: polygons, columns "name", "geometry", EPSG:4326
Count how many addresses are within each district, using a spatial join with
predicate "within". Make sure both layers have the same CRS first (raise an
error otherwise). Keep districts with no addresses, with a count of 0. Return
a GeoDataFrame with the columns "name", "addresses", "geometry".
```

**Example response** (yours may differ):

In [ ]:
assert addresses.crs == city_districts.crs, "CRS of the two layers differ"

# attach the district name to each address point
addresses_with_district = addresses.sjoin(
    city_districts[["name", "geometry"]],
    how="left",
    predicate="within"
)

# count addresses per district
counts = (
    addresses_with_district
    .groupby("name")
    .size()
    .rename("addresses")
    .reset_index()
)

# join the counts back to all districts, so that empty districts are kept
district_counts = city_districts[["name", "geometry"]].merge(counts, on="name", how="left")
district_counts["addresses"] = district_counts["addresses"].fillna(0).astype(int)
district_counts = district_counts[["name", "addresses", "geometry"]]

district_counts.drop(columns="geometry")

**Before you run it**, read it through:

- The `assert` stops the code if the two layers don’t share a CRS, as we
  requested. (We know they are both in EPSG:4326, but now the code checks it.)
- `how="left"` keeps *all addresses*, also those outside every district (their
  `name` is empty).
- The second join (`merge()`) is a normal, *attribute* join on the column
  `name`. Starting from `city_districts` with `how="left"` keeps *all
  districts*, also those without addresses.

## Check the result

Every address is either counted in exactly one district, or it lies outside
all of them. And for the *Eteläinen* district, we know the answer from the
lesson’s [point-in-polygon query](point-in-polygon-queries):

In [ ]:
outside = addresses_with_district["name"].isna().sum()
assert district_counts["addresses"].sum() + outside == len(addresses)

southern_district = city_districts[city_districts["name"] == "Eteläinen"]
in_southern_district = addresses.within(southern_district.geometry.iloc[0]).sum()

assert district_counts.loc[district_counts["name"] == "Eteläinen", "addresses"].item() == in_southern_district

print(f"{outside} address(es) outside all districts; Eteläinen: {in_southern_district} addresses, as in the lesson.")

Looks good! Finally, a quick map to see whether the result is plausible:

In [ ]:
ax = district_counts.plot(
    column="addresses",
    cmap="Reds",
    edgecolor="grey",
    legend=True,
    legend_kwds={"label": "Number of addresses"}
)
addresses.plot(ax=ax, color="black", markersize=5)
ax.set_axis_off()

## AI-use log

Whenever you use an AI assistant in this course (in exercises, only where a
task is marked **AI-LLM OK**, see the [course guidelines on AI
tools](https://autogis-site.readthedocs.io/en/latest/course-info/ai-tools.html)),
keep a short log of what you did. For this notebook, it could look like this:

| Step | What I asked | What I changed | How I checked it |
|------|--------------|----------------|------------------|
| 2 | Count addresses per district with a spatial join (`within`), keeping empty districts | Nothing | Shared only column names, data types and CRS, no data; checked that counts add up; compared Eteläinen with the point-in-polygon query |